In [81]:
import numpy as np
import wfdb
import pywt

import cv2
from scipy import signal
import os
from collections import deque
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

In [2]:
class ECGDataset(Dataset):
    def __init__(self, data_dir, prefix_dwt='X_dwt_', prefix_stft='X_stft_', prefix_y='y_'):
        self.data_dir = data_dir
        
        # Собираем список всех файлов батчей
        self.dwt_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_dwt)])
        self.stft_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_stft)])
        self.y_files = sorted([f for f in os.listdir(data_dir) if f.startswith(prefix_y)])
        
        self.cumulative_sizes = []
        current_total = 0
        
        # Индексируем датасет (считаем, сколько окон в каждом файле)
        for f in self.y_files:
            labels = np.load(os.path.join(data_dir, f))
            current_total += len(labels)
            self.cumulative_sizes.append(current_total)
        
        self.total_size = current_total
        # Кэш для последнего загруженного файла (чтобы не читать диск на каждом шаге)
        self.current_batch_idx = -1
        self.batch_data = (None, None, None)

    def __len__(self):
        return self.total_size

    def _load_batch_for_index(self, global_idx):
        # Определяем, в каком файле лежит этот индекс
        batch_idx = next(i for i, size in enumerate(self.cumulative_sizes) if size > global_idx)
        
        # Если файл уже в памяти, ничего не делаем
        if batch_idx == self.current_batch_idx:
            return batch_idx
            
        # Загружаем новые данные
        dwt = np.load(os.path.join(self.data_dir, self.dwt_files[batch_idx]))
        stft = np.load(os.path.join(self.data_dir, self.stft_files[batch_idx]))
        y = np.load(os.path.join(self.data_dir, self.y_files[batch_idx]))
        
        self.batch_data = (dwt, stft, y)
        self.current_batch_idx = batch_idx
        return batch_idx

    def __getitem__(self, idx):
        batch_idx = self._load_batch_for_index(idx)
        
        # Вычисляем локальный индекс внутри батча
        start_idx = self.cumulative_sizes[batch_idx - 1] if batch_idx > 0 else 0
        local_idx = idx - start_idx
        
        d_sample, s_sample, y_sample = self.batch_data
        
        # Извлекаем данные
        x_dwt = d_sample[local_idx]   # (125, 120, 8)
        x_stft = s_sample[local_idx]  # (129, 21, 8)
        target = y_sample[local_idx]  # One-hot вектор
        
        # Переставляем оси под PyTorch: (H, W, C) -> (C, H, W)
        x_dwt = torch.from_numpy(x_dwt).permute(2, 0, 1).float()
        x_stft = torch.from_numpy(x_stft).permute(2, 0, 1).float()
        target = torch.from_numpy(target).float()
        
        return x_dwt, x_stft, target


In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return F.relu(out)

class SEAttention(nn.Module):
    """ Простой механизм внимания для взвешивания каналов """
    def __init__(self, channels, reduction=4):
        super(SEAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class ECGMultiModalNet(nn.Module):
    def __init__(self, num_classes):
        super(ECGMultiModalNet, self).__init__()
        
        # Ветка DWT
        self.dwt_branch = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=(3, 7), padding=(1, 3)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Ветка STFT
        self.stft_branch = nn.Sequential(
            nn.Conv2d(8, 32, kernel_size=(3, 3), padding=(1, 1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            ResidualBlock(32, 32),
            SEAttention(32),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # Общий классификатор
        self.classifier = nn.Sequential(
            nn.Linear(32 + 32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_dwt, x_stft):
        feat_dwt = self.dwt_branch(x_dwt)
        feat_stft = self.stft_branch(x_stft)
        
        combined = torch.cat((feat_dwt, feat_stft), dim=1)
        return self.classifier(combined)

In [4]:
def predict_single_window(model, x_dwt, x_stft, device, threshold=0.5):
    """
    x_dwt: тензор [8, H, W]
    x_stft: тензор [8, H, W]
    """
    model.eval()
    
    # Добавляем размер батча (batch dimension), так как модель ждет [1, 8, H, W]
    x_dwt = x_dwt.unsqueeze(0).to(device)
    x_stft = x_stft.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(x_dwt, x_stft)
        
        probs = torch.sigmoid(logits).squeeze(0)
        
        predictions = (probs > threshold).int()
        
    return probs.cpu().numpy(), predictions.cpu().numpy()

In [5]:
class EKGInferenceAggregator:
    def __init__(self, window_buffer_size=10, num_classes=12):
        # Очередь для хранения векторов вероятностей (после Sigmoid)
        self.buffer = deque(maxlen=window_buffer_size)
        self.num_classes = num_classes

    def add_prediction(self, prob_vector):
        # prob_vector: выход модели после Sigmoid, размерность (12,)
        self.buffer.append(prob_vector)

    def get_max_prediction(self, threshold=0.3):
        # Max-Confidence Pooling. Берем максимальную уверенность по каждому классу среди всех окон.
        if not self.buffer:
            return np.zeros(self.num_classes), np.zeros(self.num_classes)
        
        max_probs = np.max(self.buffer, axis=0)
        binary_output = (max_probs > threshold).astype(int)
        return max_probs, binary_output

In [6]:
def window_slide(data, wind_len=2500, slide=500):
    starts = np.arange(0, max(1, len(data) - wind_len + slide), slide)
    all_data = []
    
    for s in starts:
        end = min(s + wind_len, len(data))
        start = s if s + wind_len <= len(data) else len(data) - wind_len
        all_data.append(data[start:start + wind_len])
        
    return np.array(all_data)[..., :8]

In [13]:
def get_dwt_scalogram(window, wavelet='db4', level=5, shape=(125, 120)):
    h, w = shape
    ecg_data = window.T
    all_scalograms = []
    for lead_signal in ecg_data:

        coeffs = pywt.wavedec(lead_signal, wavelet, level=level)
        stack = [cv2.resize(np.abs(c).reshape(1, -1), (w, 1))[0] for c in coeffs[::-1]]
        scalogram = np.vstack(stack)
        if scalogram.shape[0] != h:
            scalogram = cv2.resize(scalogram, (w, h), interpolation=cv2.INTER_LINEAR)

        all_scalograms.append(np.log1p(scalogram))
    multi_channel_input = np.stack(all_scalograms, axis=-1)
    return multi_channel_input

def prepare_stft_data(window, fs=500, nperseg=256, noverlap=128):
    ecg_data = window.T
    all_spectrograms = []
    for lead_signal in ecg_data:
        
        f, t, Zxx = signal.stft(lead_signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
        amplitude = np.abs(Zxx)
        
        spec_log = np.log10(amplitude + 1e-6)
        spec_norm = (spec_log - np.mean(spec_log)) / (np.std(spec_log) + 1e-8)
        all_spectrograms.append(spec_norm)
    multi_channel_input = np.stack(all_spectrograms, axis=-1)
    
    return multi_channel_input

def normalize_signal(signal):
    mean = np.mean(signal)
    std = np.std(signal) + 1e-6 # чтобы не делить на 0
    return (signal - mean) / std

def onehot_group(target):
    oh = np.zeros(12)
    st_t_codes = {'428750005', '164930006', '164934002', '429622005'} # Изменения ST/T
    rare_imp = {'111975006', '164909002', '195042002', '54329005'} # QT, БЛНПГ, и т.д.

    for code in target:
        # 0. Норма
        if code == '427084000' or code == '164865005': 
            oh[0] = 1
        # 1. Изменения ST-T
        elif code in st_t_codes:
            oh[1] = 1
        # 2. Мерцалка/Трепетание (AF/AFL)
        elif code == '164889003' or code == '164890007':
            oh[2] = 1
        # 3. Брадикардия
        elif code == '164861001':
            oh[3] = 1
        # 4. Желудочковая экстрасистолия (PVC)
        elif code == '427172004':
            oh[4] = 1
        # 5. Предсердная экстрасистолия (PAC)
        elif code == '284470004':
            oh[5] = 1
        # 6. Блокада левой ножки (LBBB)
        elif code == '164909002':
            oh[6] = 1
        # 7. Блокада правой ножки (RBBB)
        elif code == '164867002' or code == '164931005':
            oh[7] = 1
        # 8. Отклонение оси влево (LAD)
        elif code == '164873001':
            oh[8] = 1
        # 9. АВ-блокада 1 степени
        elif code == '270492004':
            oh[9] = 1
        # 10. Тахикардии
        elif code == '426177001' or code == '426761007':
            oh[10] = 1
        # 11. Rare Important (Опасные/Важные редкие)
        elif code in rare_imp:
            oh[11] = 1
            
    return oh

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load('last_checkpoint.pth')
loaded_model = ECGMultiModalNet(num_classes=12).to(device)
loaded_model.load_state_dict(checkpoint['model_state_dict'])

C:\Users\HundeRob0t\AppData\Local\Temp\ipykernel_12212\1863134475.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load('last_checkpoint.pth')


<All keys matched successfully>

In [10]:
train_dataset = ECGDataset(data_dir='D:/processed_data/train')
val_dataset = ECGDataset(data_dir='D:/processed_data/val')

train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, # для качества True, для скорости False (связано с чтением файлов и подачей их на обучение)
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

Пример с записью из другого датасета Physionet Challenge 2020 с частотой дискр. 500

In [78]:
path = "\\ptb-xl\\g1\\"
record_1 = wfdb.rdrecord(path+'HR00026') 

data_1 = record_1.p_signal
header_1 = record_1.comments

print(f"Длина записи: {data_1[:,:8].shape[0]}")

Длина записи: 5000


In [79]:
window_size = 2500
slide = 100
cut_data = window_slide(data_1, window_size, slide)

target = onehot_group(header_1[2][4:].split(","))
target = torch.from_numpy(target).float()

aggregator = EKGInferenceAggregator()

for window in cut_data:
    window = normalize_signal(window)
    d = get_dwt_scalogram(window)
    s = prepare_stft_data(window)
    d = torch.from_numpy(d).permute(2, 0, 1).float()
    s = torch.from_numpy(s).permute(2, 0, 1).float()
    prob, _ = predict_single_window(loaded_model, d, s, device, 0.2)
    aggregator.add_prediction(prob)
    
probabilities, binary_output = aggregator.get_max_prediction(threshold=0.3)
print(f"Вероятности классов: {np.round(probabilities,3)}")
print(f"Предсказанные индексы диагнозов: {torch.where(torch.tensor(binary_output) == 1)[0].tolist()}")
print(f"Реальные индексы диагнозов: {torch.where(target == 1)[0].tolist()}")

# probabilities, binary_output = aggregator.get_voted_prediction(threshold=0.3)
# print(f"Вероятности классов: {np.round(probabilities,3)}")
# print(f"Предсказанные индексы диагнозов: {torch.where(torch.tensor(binary_output) == 1)[0].tolist()}")
# print(f"Реальные индексы диагнозов: {torch.where(target == 1)[0].tolist()}")

Вероятности классов: [0.606 0.639 0.01  0.094 0.148 0.145 0.005 0.152 0.089 0.178 0.003 0.068]
Предсказанные индексы диагнозов: [0, 1]
Реальные индексы диагнозов: [1]


In [82]:
def validate_model_with_aggregator(model, val_loader, device, threshold=0.3, n_windows=4):
    """
    Валидация с использованием Max-Pooling агрегатора.
    n_windows: сколько последовательных окон объединяем в один прогноз.
    """
    model.eval()
    model = model.to(device)
    
    all_agg_preds = []
    all_agg_labels = []
    total_loss = 0.0
    criterion = torch.nn.BCEWithLogitsLoss()

    # Временные буферы для агрегации
    temp_probs = []
    temp_labels = []

    with torch.no_grad():
        for x_dwt, x_stft, labels in val_loader:
            x_dwt, x_stft, labels = x_dwt.to(device), x_stft.to(device), labels.to(device)
            
            logits = model(x_dwt, x_stft)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            # Получаем вероятности
            probs = torch.sigmoid(logits).cpu().numpy()
            labels_np = labels.cpu().numpy()

            # Идем по батчу и наполняем агрегатор
            for i in range(len(probs)):
                temp_probs.append(probs[i])
                temp_labels.append(labels_np[i])

                # Как только накопили n_windows, схлопываем их в один результат
                if len(temp_probs) == n_windows:
                    # MAX POOLING: Берем максимум по каждому классу
                    agg_prob = np.max(temp_probs, axis=0)
                    agg_pred = (agg_prob > threshold).astype(float)
                    
                    # Для меток: если хоть в одном окне была болезнь, считаем, что она есть в записи
                    agg_label = np.max(temp_labels, axis=0)
                    
                    all_agg_preds.append(agg_pred)
                    all_agg_labels.append(agg_label)
                    
                    # Очищаем буфер для следующей группы
                    temp_probs = []
                    temp_labels = []

    # Если в конце остались окна, которые не собрались в полный набор n_windows
    if temp_probs:
        all_agg_preds.append((np.max(temp_probs, axis=0) > threshold).astype(float))
        all_agg_labels.append(np.max(temp_labels, axis=0))

    all_agg_preds = np.vstack(all_agg_preds)
    all_agg_labels = np.vstack(all_agg_labels)
    
    metrics = {
        'val_loss': total_loss / len(val_loader),
        'f1_macro': f1_score(all_agg_labels, all_agg_preds, average='macro', zero_division=0),
        'f1_micro': f1_score(all_agg_labels, all_agg_preds, average='micro', zero_division=0),
        'precision': precision_score(all_agg_labels, all_agg_preds, average='macro', zero_division=0),
        'recall': recall_score(all_agg_labels, all_agg_preds, average='macro', zero_division=0)
    }

    print(f"\n--- Aggregated Validation Results (Max-Pooling, n={n_windows}) ---")
    print(f"Loss: {metrics['val_loss']:.4f}")
    print(f"F1 Macro: {metrics['f1_macro']:.4f} | F1 Micro: {metrics['f1_micro']:.4f}")
    print(f"Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f}")
    print(classification_report(all_agg_labels, all_agg_preds, zero_division=0))
    
    return metrics
validate_model_with_aggregator(loaded_model, val_loader, device, threshold=0.4, n_windows=10)


--- Aggregated Validation Results (Max-Pooling, n=10) ---
Loss: 0.2341
F1 Macro: 0.4356 | F1 Micro: 0.6943
Precision: 0.4006 | Recall: 0.4997
              precision    recall  f1-score   support

           0       0.69      0.88      0.78        57
           1       0.95      0.98      0.96        91
           2       0.72      0.78      0.75        23
           3       0.40      0.66      0.50        29
           4       0.20      0.12      0.15        17
           5       0.00      0.00      0.00         9
           6       0.00      0.00      0.00         1
           7       0.80      0.93      0.86        74
           8       0.22      0.62      0.32        16
           9       0.22      0.20      0.21        10
          10       0.30      0.43      0.35         7
          11       0.30      0.40      0.34        15

   micro avg       0.63      0.77      0.69       349
   macro avg       0.40      0.50      0.44       349
weighted avg       0.66      0.77      0.70  

{'val_loss': 0.23409708638985952,
 'f1_macro': 0.43557578241986805,
 'f1_micro': 0.694300518134715,
 'precision': 0.4006206151830896,
 'recall': 0.4997205824792323}